In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from typing import Dict, List

def scrape_hadith(hadith_num: int) -> Dict:
    """Scrape hadith from islam-db.com"""

    # URL pattern - you'll need to figure out the chapter ID (97663 in your example)
    # For now, let's try with direct hadith number
    url = f"https://hadith.islam-db.com/single-book/146/صحيح-البخاري/97663/{hadith_num}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch hadith {hadith_num}: {e}")
        return None

    soup = BeautifulSoup(response.content, 'html.parser')

    # Extract isnad with narrators
    hadith_text = soup.find('p', class_='more-height')
    if not hadith_text:
        return None

    # Extract narrator links from isnad
    narrator_links = hadith_text.find_all('a', class_='rawy')

    chain = []
    for link in narrator_links:
        narrator_id = link.get('id')
        narrator_name = link.text.strip()
        chain.append({
            'name': narrator_name,
            'id': narrator_id
        })

    # Extract narrator details from table
    narrator_table = soup.find('table', class_='table-striped')
    narrator_details = []

    if narrator_table:
        rows = narrator_table.find_all('tr')[1:]  # Skip header
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                name_cell = cols[0]
                link = name_cell.find('a')

                narrator_details.append({
                    'name': link.text.strip() if link else '',
                    'id': link.get('data-id') if link else '',
                    'fame': cols[1].text.strip(),
                    'rank': cols[2].text.strip()
                })

    # Extract matn
    matn_tag = hadith_text.find('a', class_='matn')
    matn = matn_tag.text.strip() if matn_tag else ""

    # Full text
    full_text = hadith_text.get_text(strip=True)

    return {
        'hadith_number': hadith_num,
        'url': url,
        'chain': chain,
        'narrator_details': narrator_details,
        'matn': matn,
        'full_text': full_text
    }

def scrape_bukhari_range(start: int, end: int, output_file: str = 'golden_dataset.json'):
    """Scrape range of hadiths and save"""

    results = []

    for num in range(start, end + 1):
        print(f"Scraping hadith {num}...")
        hadith = scrape_hadith(num)

        if hadith:
            results.append(hadith)
            print(f"  ✓ Got {len(hadith['chain'])} narrators")
        else:
            print(f"  ✗ Failed")

        # Be nice to the server
        time.sleep(1)

    # Save
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Scraped {len(results)} hadiths → {output_file}")

In [74]:
hadith = scrape_hadith(2)

In [ ]:
https://hadith.islam-db.com/narrators/4049/%D8%B9%D8%A7%D8%A6%D8%B4%D8%A9-%D8%A8%D9%86%D8%AA-%D8%B9%D8%A8%D8%AF-%D8%A7%D9%84%D9%84%D9%87-%D8%A8%D9%86-%D8%B9%D8%AB%D9%85%D8%A7%D9%86-%D8%A8%D9%86...

In [75]:
hadith

{'hadith_number': 2,
 'url': 'https://hadith.islam-db.com/single-book/146/صحيح-البخاري/97663/2',
 'chain': [{'name': 'عَبْدُ اللَّهِ بْنُ يُوسُفَ', 'id': '5175'},
  {'name': 'مَالِكٌ', 'id': '6659'},
  {'name': 'هِشَامِ بْنِ عُرْوَةَ', 'id': '8055'},
  {'name': 'أَبِيهِ', 'id': '5594'},
  {'name': 'عَائِشَةَ', 'id': '4049'}],
 'narrator_details': [{'name': 'عَائِشَةَ',
   'id': '4049',
   'fame': 'عائشة بنت أبي بكر الصديق                                                                                                    / توفي في :57',
   'rank': 'صحابي'},
  {'name': 'أَبِيهِ',
   'id': '5594',
   'fame': 'عروة بن الزبير الأسدي                                                                                                    / توفي في :94',
   'rank': 'ثقة فقيه مشهور'},
  {'name': 'هِشَامِ بْنِ عُرْوَةَ',
   'id': '8055',
   'fame': 'هشام بن عروة الأسدي                                                   / ولد في :58                                                    / توفي في :145',
   '

In [1]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

In [2]:
def check_null_names():
    with driver.session() as session:
        result = session.run("""
            MATCH (n:Person)
            WHERE n.name IS NULL
            RETURN n.id, n.fame
            LIMIT 10
        """)

        print("\nPersons with NULL names:")
        for record in result:
            print(f"  ID: {record['n.id']}, Fame: {record['n.fame']}")

check_null_names()

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: id)", position=<SummaryInputPosition line=4, column=22, offset=84>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'column': 22, 'offset': 84, 'line': 4}}> for query: '\n            MATCH (n:Person)\n            WHERE n.name IS NULL\n            RETURN n.id, n.fame\n            LIMIT 10\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_descriptio


Persons with NULL names:


In [16]:
def debug_hadith_2_edges():
    """Check what edges exist for hadith 2"""

    with driver.session() as session:
        # Check all edges with hadith=2
        result = session.run("""
            MATCH (n1:Person)-[r:NARRATED_FROM {hadith: 2}]->(n2:Person)
            RETURN n1.name as from_narrator,
                   n1.id as from_id,
                   n2.name as to_narrator,
                   n2.id as to_id
            ORDER BY from_narrator
        """)

        print("All NARRATED_FROM edges for hadith 2:")
        for record in result:
            print(f"  {record['from_narrator']} ({record['from_id']}) -> {record['to_narrator']} ({record['to_id']})")

debug_hadith_2_edges()

# Also check what the scraped data says
import json

with open('bukhari_hadiths.json', 'r', encoding='utf-8') as f:
    hadiths = json.load(f)

hadith_2 = next(h for h in hadiths if h['hadith_number'] == 2)

print("\nExpected chain from scraped data:")
for i in range(len(hadith_2['chain']) - 1):
    n1 = hadith_2['chain'][i]
    n2 = hadith_2['chain'][i + 1]
    print(f"  {n1['name']} ({n1['id']}) -> {n2['name']} ({n2['id']})")

All NARRATED_FROM edges for hadith 2:
  عَبْدُ اللَّهِ بْنُ يُوسُفَ (5175) -> مَالِكٌ (6659)
  عُرْوَةَ بْنَ الزُّبَيْرِ (5594) -> عَائِشَةُ (4049)
  مَالِكٌ (6659) -> هِشَامٍ (8055)
  هِشَامٍ (8055) -> عُرْوَةَ بْنَ الزُّبَيْرِ (5594)

Expected chain from scraped data:
  عَبْدُ اللَّهِ بْنُ يُوسُفَ (5175) -> مَالِكٌ (6659)
  مَالِكٌ (6659) -> هِشَامِ بْنِ عُرْوَةَ (8055)
  هِشَامِ بْنِ عُرْوَةَ (8055) -> أَبِيهِ (5594)
  أَبِيهِ (5594) -> عَائِشَةَ (4049)


In [17]:
import json

with open('bukhari_hadiths.json', 'r', encoding='utf-8') as f:
    hadiths = json.load(f)

hadith_2 = next(h for h in hadiths if h['hadith_number'] == 2)
print("\nScraped chain for hadith 2:")
for i, n in enumerate(hadith_2['chain']):
    print(f"  {i+1}. {n['name']} (ID: {n['id']})")


Scraped chain for hadith 2:
  1. عَبْدُ اللَّهِ بْنُ يُوسُفَ (ID: 5175)
  2. مَالِكٌ (ID: 6659)
  3. هِشَامِ بْنِ عُرْوَةَ (ID: 8055)
  4. أَبِيهِ (ID: 5594)
  5. عَائِشَةَ (ID: 4049)


In [18]:
def verify_chain_by_id():
    """Check if chain is complete by following IDs"""

    with driver.session() as session:
        result = session.run("""
            MATCH (h:Hadith {number: 2})-[:HAS_CHAIN]->(first:Person {id: '5175'})
            MATCH path = (first)-[:NARRATED_FROM*]->(last:Person)
            WHERE ALL(r IN relationships(path) WHERE r.hadith = 2)
            RETURN [node in nodes(path) | node.id + ':' + node.name] as chain,
                   length(path) as len
            ORDER BY len DESC
            LIMIT 1
        """)

        for record in result:
            print(f"Chain length: {record['len']}")
            print("Chain by ID:name:")
            for node in record['chain']:
                print(f"  -> {node}")

verify_chain_by_id()

Chain length: 4
Chain by ID:name:
  -> 5175:عَبْدُ اللَّهِ بْنُ يُوسُفَ
  -> 6659:مَالِكٌ
  -> 8055:هِشَامٍ
  -> 5594:عُرْوَةَ بْنَ الزُّبَيْرِ
  -> 4049:عَائِشَةُ


In [5]:
def final_validation_optimized():
    """Validate with memory-efficient queries"""

    with driver.session() as session:

        print("="*60)
        print("FINAL GRAPH VALIDATION")
        print("="*60)

        # Check a sample of chains (not all at once)
        print(f"\n📊 Sample Chain Lengths (first 100 hadiths):")
        result = session.run("""
            MATCH (h:Hadith)-[:HAS_CHAIN]->(first:Person)
            WHERE h.number <= 100
            MATCH path = (first)-[:NARRATED_FROM*]->(last:Person)
            WHERE ALL(r IN relationships(path) WHERE r.hadith = h.number)
              AND NOT EXISTS((last)-[:NARRATED_FROM {hadith: h.number}]->())
            RETURN AVG(length(path)) as avg_length,
                   MIN(length(path)) as min_length,
                   MAX(length(path)) as max_length
        """)

        for record in result:
            print(f"   Average: {record['avg_length']:.1f}")
            print(f"   Range: {record['min_length']} - {record['max_length']}")

        # Most common narrators (simpler query)
        print(f"\n👥 Top 10 Narrators by Total Connections:")
        result = session.run("""
            MATCH (n:Person)
            OPTIONAL MATCH (n)-[:NARRATED_FROM]->()
            WITH n, COUNT(*) as out_count
            OPTIONAL MATCH ()-[:NARRATED_FROM]->(n)
            WITH n, out_count, COUNT(*) as in_count
            RETURN n.name, n.rank, (out_count + in_count) as total
            ORDER BY total DESC
            LIMIT 10
        """)

        for record in result:
            print(f"   {record['n.name']}: {record['total']} connections ({record['n.rank']})")

        # Sample chains
        print(f"\n🔗 Sample Chains:")
        for hadith_num in [1, 2, 50, 100]:
            result = session.run("""
                MATCH (h:Hadith {number: $num})-[:HAS_CHAIN]->(first:Person)
                MATCH path = (first)-[:NARRATED_FROM*]->(last:Person)
                WHERE ALL(r IN relationships(path) WHERE r.hadith = $num)
                  AND NOT EXISTS((last)-[:NARRATED_FROM {hadith: $num}]->())
                RETURN [node in nodes(path) | node.name] as chain
                LIMIT 1
            """, num=hadith_num)

            for record in result:
                chain_str = ' → '.join(filter(None, record['chain']))
                print(f"   Hadith {hadith_num}: {chain_str[:80]}...")

        # Totals
        print(f"\n📈 Database Totals:")
        result = session.run("MATCH (h:Hadith) RETURN count(h) as count")
        print(f"   Hadiths: {result.single()['count']}")

        result = session.run("MATCH (n:Person) RETURN count(n) as count")
        print(f"   Narrators: {result.single()['count']}")

        result = session.run("MATCH ()-[r:NARRATED_FROM]->() RETURN count(r) as count")
        print(f"   Transmission edges: {result.single()['count']}")

final_validation_optimized()

FINAL GRAPH VALIDATION

📊 Sample Chain Lengths (first 100 hadiths):
   Average: 4.5
   Range: 2 - 14

👥 Top 10 Narrators by Total Connections:
   الزُّهْرِيِّ: 2646 connections (الفقيه الحافظ متفق على جلالته وإتقانه)
   شُعْبَةُ: 1674 connections (ثقة حافظ متقن عابد)
   مَالِكٌ: 1315 connections (رأس المتقنين وكبير المتثبتين)
   عُرْوَةَ بْنَ الزُّبَيْرِ: 1272 connections (ثقة فقيه مشهور)
   أَبِي هُرَيْرَةَ: 1258 connections (صحابي)
   أَنَسٌ: 1024 connections (صحابي)
   عَائِشَةُ: 989 connections (صحابي)
   ابْنِ عُمَرَ: 956 connections (صحابي)
   اللَّيْثُ: 903 connections (ثقة ثبت فقيه إمام مشهور)
   لِابْنِ عَبَّاسٍ: 875 connections (صحابي)

🔗 Sample Chains:
   Hadith 1: الْحُمَيْدِيُّ → سُفْيَانُ → يَحْيَى → مُحَمَّدِ بْنِ إِبْرَاهِيمَ → وَعَلْقَمَة...
   Hadith 2: عَبْدُ اللَّهِ بْنُ يُوسُفَ → مَالِكٌ → هِشَامٍ → عُرْوَةَ بْنَ الزُّبَيْرِ → عَ...
   Hadith 50: إِبْرَاهِيمُ بْنُ حَمْزَةَ → إِبْرَاهِيمُ بْنُ سَعْدٍ → صَالِحٍ → الزُّهْرِيِّ →...
   Hadith 100: آدَمُ → شُعْبَةُ → عَ

In [1]:
import re

# Arabic diacritics (tashkeel)
ARABIC_DIACRITICS = re.compile(r"""
    ّ    | # Shadda
    َ    | # Fatha
    ً    | # Tanwin Fath
    ُ    | # Damma
    ٌ    | # Tanwin Damm
    ِ    | # Kasra
    ٍ    | # Tanwin Kasr
    ْ    | # Sukun
    ـ      # Tatweel
""", re.VERBOSE)

def remove_tashkeel(text: str) -> str:
    return re.sub(ARABIC_DIACRITICS, '', text)


text = "هِشَامِ بْنِ عُرْوَةَ"
print(remove_tashkeel(text))


هشام بن عروة


In [ ]:
import unicodedata

def remove_diacritics(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

text = "هِشَامِ بْنِ عُرْوَةَ"
print(remove_diacritics(text))

هشام بن عروة
